# Complementos de Investigação Operacional — capítulo 5: Programação dinâmica
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/notebooks/05_dynamic_programming.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 5 of the course "Complements of Operational Research" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/applied-operations-research-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 5.1 Programação dinâmica: princípio de Bellman e caminho mais curto

Exemplo [`ex05_1_bellman_shortest_path.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/05_dynamic_programming/05_1_bellman_shortest_path/ex05_1_bellman_shortest_path.py) · funções da UC: [`caminho.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/05_dynamic_programming/05_1_bellman_shortest_path/caminho.py)

```text
Reproduz os exemplos do deck 5.1 (princípio de Bellman e caminho mais curto).

Exemplo-guia: transportadora de A a J (custos em centenas de €).
Regra gulosa: A-C-F-I-J, custo 21. Recursão para trás: f(H) = 5, f(I) = 7; f(E) = 11 (H ou I),
f(F) = 16, f(G) = 9; f(B) = 13, f(C) = 18 (E ou G), f(D) = 13; f(A) = 17. Ótimo A-B-G-I-J, custo 17.
Slide «Quanto se poupa?»: 15 percursos, 45 somas a enumerar contra 19 da PD; tabela k^N contra N k^2.
Para resolver na aula: Exemplos 1, 2 e 3 (do nó 1 ao nó 7): 26 por 1-4-6-7; 23 por 1-3-6-7;
17 por 1-2-3-5-7 (a regra gulosa dá 34, 28 e 17).

No fim compara com os slides.
```

In [2]:
from caminho import caminho_etapas, caminhos_otimos, todos_caminhos, guloso


def txt(caminho):
    """Percurso como texto: A-B-G-I-J."""
    return "-".join(map(str, caminho))


def mostra(titulo, arcos, ini, fim):
    """Regra gulosa, recursão para trás nó a nó, percursos ótimos e força bruta."""
    print("\n" + titulo)
    g, cg = guloso(arcos, ini, fim)
    print("  regra gulosa: %s, custo %d" % (txt(cg), g))
    f, dec = caminho_etapas(arcos, fim)
    print("  recursão para trás (f = custo mínimo até %s):" % fim)
    for i in sorted(arcos, reverse=True):
        vias = ", ".join("via %s: %d+%d=%d" % (j, c, f[j], c + f[j]) for j, c in arcos[i].items())
        print("    f(%s) = %-3d [%s]  -> %s" % (i, f[i], vias, " ou ".join(map(str, dec[i]))))
    cs = caminhos_otimos(dec, ini, fim)
    print("  ótimo: %d por %s" % (f[ini], " e ".join(txt(c) for c in cs)))
    tc = todos_caminhos(arcos, ini, fim)
    print("  força bruta: %d percursos, mínimo %d" % (len(tc), min(v for _, v in tc)))
    return g, f, dec, cs, tc


ok = True
print("Complementos de IO — deck 5.1: princípio de Bellman e caminho mais curto")

# ------------------------------------------------------------ transportadora A -> J
arcos = {"A": {"B": 4, "C": 2, "D": 8}, "B": {"E": 4, "F": 2, "G": 4}, "C": {"E": 7, "F": 3, "G": 9},
         "D": {"E": 9, "F": 7, "G": 4}, "E": {"H": 6, "I": 4}, "F": {"I": 9}, "G": {"H": 8, "I": 2},
         "H": {"J": 5}, "I": {"J": 7}}
g, f, dec, cs, tc = mostra("Transportadora de A a J (custos em centenas de €)", arcos, "A", "J")
somas_enum = sum(len(p) - 2 for p, _ in tc)          # um percurso com m arcos pede m - 1 somas
somas_pd = sum(len(s) for s in arcos.values())       # uma soma por arco
print("  enumerar: %d somas; PD: %d somas (uma por arco)" % (somas_enum, somas_pd))
print("  em geral (N etapas, k nós por etapa):")
esforco = [(N, k, float(k) ** N, N * k * k) for N, k in [(4, 3), (10, 10), (20, 10)]]
for N, k, perc, somas in esforco:
    print("    N = %2d, k = %2d: %g percursos, %g somas da PD" % (N, k, perc, somas))
ok &= (g == 21 and f["A"] == 17 and cs == [["A", "B", "G", "I", "J"]]
       and [f[n] for n in "BCDEFGHI"] == [13, 18, 13, 11, 16, 9, 5, 7]
       and dec["E"] == ["H", "I"] and dec["C"] == ["E", "G"]
       and len(tc) == 15 and somas_enum == 45 and somas_pd == 19
       and [(p, s) for _, _, p, s in esforco] == [(81, 36), (1e10, 1000), (1e20, 2000)])

# ------------------------------------------------------------ Para resolver na aula
E1 = {1: {2: 5, 3: 9, 4: 8}, 2: {5: 10, 6: 17}, 3: {5: 4, 6: 10}, 4: {5: 9, 6: 9}, 5: {7: 19}, 6: {7: 9}}
E2 = {1: {2: 7, 3: 8, 4: 9}, 2: {5: 12}, 3: {5: 8, 6: 9}, 4: {5: 7, 6: 13}, 5: {7: 9}, 6: {7: 6}}
E3 = {1: {2: 5, 3: 14, 4: 6}, 2: {3: 6, 5: 12, 6: 12}, 3: {5: 2, 6: 3}, 4: {3: 7, 6: 9}, 5: {7: 4}, 6: {7: 4}}
esperado = [(26, [[1, 4, 6, 7]], 34), (23, [[1, 3, 6, 7]], 28), (17, [[1, 2, 3, 5, 7]], 17)]
for n, (G, (fo, co, go)) in enumerate(zip([E1, E2, E3], esperado), start=1):
    g, f, dec, cs, tc = mostra("Exemplo %d (do nó 1 ao nó 7)" % n, G, 1, 7)
    ok &= f[1] == fo and cs == co and g == go and min(v for _, v in tc) == fo
ok &= dec[4] == [3, 6]                                # Exemplo 3: empate em 4, que não interessa ao ótimo

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 5.1: princípio de Bellman e caminho mais curto

Transportadora de A a J (custos em centenas de €)
  regra gulosa: A-C-F-I-J, custo 21
  recursão para trás (f = custo mínimo até J):
    f(I) = 7   [via J: 7+0=7]  -> J
    f(H) = 5   [via J: 5+0=5]  -> J
    f(G) = 9   [via H: 8+5=13, via I: 2+7=9]  -> I
    f(F) = 16  [via I: 9+7=16]  -> I
    f(E) = 11  [via H: 6+5=11, via I: 4+7=11]  -> H ou I
    f(D) = 13  [via E: 9+11=20, via F: 7+16=23, via G: 4+9=13]  -> G
    f(C) = 18  [via E: 7+11=18, via F: 3+16=19, via G: 9+9=18]  -> E ou G
    f(B) = 13  [via E: 4+11=15, via F: 2+16=18, via G: 4+9=13]  -> G
    f(A) = 17  [via B: 4+13=17, via C: 2+18=20, via D: 8+13=21]  -> B
  ótimo: 17 por A-B-G-I-J
  força bruta: 15 percursos, mínimo 17
  enumerar: 45 somas; PD: 19 somas (uma por arco)
  em geral (N etapas, k nós por etapa):
    N =  4, k =  3: 81 percursos, 36 somas da PD
    N = 10, k = 10: 1e+10 percursos, 1000 somas da PD
    N = 20, k = 10: 1e+20 percursos,

## 5.2 Programação dinâmica: o problema da mochila

Exemplo [`ex05_2_knapsack.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/05_dynamic_programming/05_2_knapsack/ex05_2_knapsack.py) · funções da UC: [`mochila.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/05_dynamic_programming/05_2_knapsack/mochila.py)

```text
Reproduz os exemplos do deck 5.2 (o problema da mochila).

Exemplo 1 (camião, 10 t, mochila ilimitada): paletes A (3 t, 7), B (4 t, 13), C (6 t, 17).
Tabela f(0..10) = 0, 0, 0, 7, 13, 13, 17, 20, 26, 26, 30; ótimo uma B e uma C, lucro 30
(única solução por força bruta); pelo rácio, B, B: 26.
Exemplo 2 (projetos, orçamento 10, mochila 0-1): custos 3, 5, 3, 4, 1; benefícios 12, 14, 15, 15, 8.
Tabela f_5..f_1; ótimo P1, P3, P4, benefício 42 (confirmado por milp); pelo rácio P5, P3, P1: 35.
Para resolver na aula: Exemplo 4 (navio): 62, duas unidades do produto 1 (rácio 61);
Exemplo 5: 19, 2A + B (rácio 18); Exemplo 6: 26, A + 2C (rácio 24).

No fim compara com os slides.
```

In [3]:
import numpy as np

from mochila import mochila_ilimitada, solucoes_ilimitada, mochila_01, forca_bruta_01, regra_racio


def mat2str(A):
    """Matriz/vetor no formato do mat2str do MATLAB (para as saídas serem iguais)."""
    A = np.atleast_2d(np.asarray(A, float))
    return "[" + ";".join(" ".join("%.15g" % x for x in linha) for linha in A) + "]"


def mostra_ilimitada(titulo, w, v, W, nomes):
    """Tabela f(k), leitura da solução, força bruta e regra do rácio."""
    print("\n" + titulo)
    f, esc = mochila_ilimitada(w, v, W)
    for k in range(W + 1):
        cand = ", ".join("%s: %d+f(%d)=%d" % (nomes[i], v[i], k - w[i], v[i] + f[k - w[i]])
                         for i in range(len(w)) if w[i] <= k)
        quem = " (%s)" % " ou ".join(nomes[i] for i in esc[k]) if esc[k] else ""
        print("  k=%2d: %s -> f(%d) = %d%s" % (k, cand or "nada cabe", k, f[k], quem))
    k, leitura = W, []
    while esc[k]:                                     # ler a solução a partir de f(W)
        i = esc[k][0]
        leitura.append("pôr %s, restam %d" % (nomes[i], k - w[i]))
        k -= w[i]
    print("  leitura a partir de f(%d): %s" % (W, "; ".join(leitura)))
    melhor, sols = solucoes_ilimitada(w, v, W)
    print("  ótimo %d; soluções por força bruta (unidades de %s): %s" % (melhor, ", ".join(nomes), mat2str(sols)))
    vr, xr = regra_racio(w, v, W)
    print("  regra do rácio: %s, valor %d" % (mat2str(xr), vr))
    return f, melhor, sols, vr


ok = True
print("Complementos de IO — deck 5.2: o problema da mochila")

# ------------------------------------------------------------ Exemplo 1: camião (ilimitada)
f, melhor, sols, vr = mostra_ilimitada("Exemplo 1: carregar um camião, 10 t (mochila ilimitada)",
                                       [3, 4, 6], [7, 13, 17], 10, ["A", "B", "C"])
ok &= f == [0, 0, 0, 7, 13, 13, 17, 20, 26, 26, 30] and sols == [(0, 1, 1)] and vr == 26

# ------------------------------------------------------------ Exemplo 2: projetos (0-1)
w, v, W = [3, 5, 3, 4, 1], [12, 14, 15, 15, 8], 10
P = ["P1", "P2", "P3", "P4", "P5"]
print("\nExemplo 2: escolher projetos, orçamento 10 (mochila 0-1)")
valor, x, F = mochila_01(w, v, W, tabela=True)
for i in range(len(w) - 1, -1, -1):
    print("  f_%d: %s" % (i + 1, mat2str(F[i])))
k = W
for i in range(len(w)):                               # ler a solução para a frente
    if w[i] <= k:
        print("  %s: não f_%d(%d) = %d, sim %d+f_%d(%d) = %d -> %s"
              % (P[i], i + 2, k, F[i + 1][k], v[i], i + 2, k - w[i], v[i] + F[i + 1][k - w[i]],
                 "sim" if x[i] else "não"))
    else:
        print("  %s: não cabe" % P[i])
    k -= w[i] * x[i]
print("  ótimo %d com x = %s (custo %d)" % (valor, mat2str(x), sum(a * b for a, b in zip(w, x))))
melhor01, sols01 = forca_bruta_01(w, v, W)
print("  força bruta: %d com %s" % (melhor01, mat2str(sols01)))
vr01, xr01 = regra_racio(w, v, W, ilimitada=False)
print("  regra do rácio: %s, valor %d" % (mat2str(xr01), vr01))
try:                                                  # confirmação por programação linear inteira
    from scipy.optimize import milp, LinearConstraint
    r = milp(-np.array(v), constraints=LinearConstraint(np.array(w), 0, W),
             integrality=np.ones(len(w)), bounds=(0, 1))
    xm = np.round(r.x).astype(int)
    print("  milp (scipy): %g com %s" % (-r.fun, mat2str(xm)))
    ok &= abs(-r.fun - 42) < 1e-9 and xm.tolist() == [1, 0, 1, 1, 0]
except ImportError:
    print("  milp (scipy): scipy não está instalado; a força bruta confirma")
ok &= (valor == 42 and x == [1, 0, 1, 1, 0] and melhor01 == 42 and sols01 == [(1, 0, 1, 1, 0)]
       and vr01 == 35 and xr01 == [1, 0, 1, 0, 1]
       and F[4] == [0] + [8] * 10 and F[3] == [0, 8, 8, 8, 15, 23, 23, 23, 23, 23, 23]
       and F[2] == [0, 8, 8, 15, 23, 23, 23, 30, 38, 38, 38] and F[1] == F[2]
       and F[0] == [0, 8, 8, 15, 23, 23, 27, 35, 38, 38, 42])

# ------------------------------------------------------------ Para resolver na aula
_, m4, s4, r4 = mostra_ilimitada("Exemplo 4: o navio, 4 t", [2, 3, 1], [31, 47, 14], 4, ["1", "2", "3"])
_, m5, s5, r5 = mostra_ilimitada("Exemplo 5: mochila de 10 kg", [3, 4, 7], [6, 7, 10], 10, ["A", "B", "C"])
_, m6, s6, r6 = mostra_ilimitada("Exemplo 6: mochila de 13 kg", [3, 4, 5, 6], [6, 7, 10, 12], 13,
                                 ["A", "B", "C", "D"])
ok &= (m4, s4, r4) == (62, [(2, 0, 0)], 61) and (m5, s5, r5) == (19, [(2, 1, 0)], 18) \
    and (m6, s6, r6) == (26, [(1, 0, 2, 0)], 24)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 5.2: o problema da mochila

Exemplo 1: carregar um camião, 10 t (mochila ilimitada)
  k= 0: nada cabe -> f(0) = 0
  k= 1: nada cabe -> f(1) = 0
  k= 2: nada cabe -> f(2) = 0
  k= 3: A: 7+f(0)=7 -> f(3) = 7 (A)
  k= 4: A: 7+f(1)=7, B: 13+f(0)=13 -> f(4) = 13 (B)
  k= 5: A: 7+f(2)=7, B: 13+f(1)=13 -> f(5) = 13 (B)
  k= 6: A: 7+f(3)=14, B: 13+f(2)=13, C: 17+f(0)=17 -> f(6) = 17 (C)
  k= 7: A: 7+f(4)=20, B: 13+f(3)=20, C: 17+f(1)=17 -> f(7) = 20 (A ou B)
  k= 8: A: 7+f(5)=20, B: 13+f(4)=26, C: 17+f(2)=17 -> f(8) = 26 (B)
  k= 9: A: 7+f(6)=24, B: 13+f(5)=26, C: 17+f(3)=24 -> f(9) = 26 (B)
  k=10: A: 7+f(7)=27, B: 13+f(6)=30, C: 17+f(4)=30 -> f(10) = 30 (B ou C)
  leitura a partir de f(10): pôr B, restam 6; pôr C, restam 0
  ótimo 30; soluções por força bruta (unidades de A, B, C): [0 1 1]
  regra do rácio: [0 2 0], valor 26

Exemplo 2: escolher projetos, orçamento 10 (mochila 0-1)
  f_5: [0 8 8 8 8 8 8 8 8 8 8]
  f_4: [0 8 8 8 15 23 23 23 23 23 23]
  f_3: [0 8 8 15

  milp (scipy): 42 com [1 0 1 1 0]

Exemplo 4: o navio, 4 t
  k= 0: nada cabe -> f(0) = 0
  k= 1: 3: 14+f(0)=14 -> f(1) = 14 (3)
  k= 2: 1: 31+f(0)=31, 3: 14+f(1)=28 -> f(2) = 31 (1)
  k= 3: 1: 31+f(1)=45, 2: 47+f(0)=47, 3: 14+f(2)=45 -> f(3) = 47 (2)
  k= 4: 1: 31+f(2)=62, 2: 47+f(1)=61, 3: 14+f(3)=61 -> f(4) = 62 (1)
  leitura a partir de f(4): pôr 1, restam 2; pôr 1, restam 0
  ótimo 62; soluções por força bruta (unidades de 1, 2, 3): [2 0 0]
  regra do rácio: [0 1 1], valor 61

Exemplo 5: mochila de 10 kg
  k= 0: nada cabe -> f(0) = 0
  k= 1: nada cabe -> f(1) = 0
  k= 2: nada cabe -> f(2) = 0
  k= 3: A: 6+f(0)=6 -> f(3) = 6 (A)
  k= 4: A: 6+f(1)=6, B: 7+f(0)=7 -> f(4) = 7 (B)
  k= 5: A: 6+f(2)=6, B: 7+f(1)=7 -> f(5) = 7 (B)
  k= 6: A: 6+f(3)=12, B: 7+f(2)=7 -> f(6) = 12 (A)
  k= 7: A: 6+f(4)=13, B: 7+f(3)=13, C: 10+f(0)=10 -> f(7) = 13 (A ou B)
  k= 8: A: 6+f(5)=13, B: 7+f(4)=14, C: 10+f(1)=10 -> f(8) = 14 (B)
  k= 9: A: 6+f(6)=18, B: 7+f(5)=14, C: 10+f(2)=10 -> f(9) = 18 (A)
  k=

## 5.3 Programação dinâmica: gestão da mão-de-obra

Exemplo [`ex05_3_workforce.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/05_dynamic_programming/05_3_workforce/ex05_3_workforce.py) · funções da UC: [`mao_de_obra.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/05_dynamic_programming/05_3_workforce/mao_de_obra.py)

```text
Reproduz os exemplos do deck 5.3 (gestão da mão-de-obra).

Exemplo-guia: montagem de feiras e congressos, necessidades 3, 8, 9, 4, 6; excesso 300 €;
contratação 600 € + 150 € por operário; ninguém contratado no início.
Seguir as necessidades: 4050 €; 9 desde o início: 6450 €; 336 planos possíveis.
Tabelas: f_5(4) = 900, f_5(5) = 750, f_5(6..9) = 0; f_4(9) = 600 (x_4 = 6); f_3(8) = 1350, f_3(9) = 600;
f_2(3) = 2400 (x_2 = 9); f_1(0) = 3450. Plano ótimo 3, 9, 9, 6, 6 (custos 1050, 1800, 0, 600, 0),
único por força bruta. Sensibilidade ao custo fixo: 0 -> 1650; 300 -> 2850 (4 planos);
600 -> 3450; 1200 -> 4650; 2000 -> 6050 (9, 9, 9, 6, 6).
Para resolver na aula: Exemplo 7: 5, 8, 8, 6, 6, custo 3300; Exemplo 8: 5, 7, 6, 6, custo 2400.

No fim compara com os slides.
```

In [4]:
import numpy as np

from mao_de_obra import mao_obra, mao_obra_bruta, custo_plano


def mat2str(A):
    """Matriz/vetor no formato do mat2str do MATLAB (para as saídas serem iguais)."""
    A = np.atleast_2d(np.asarray(A, float))
    return "[" + ";".join(" ".join("%.15g" % x for x in linha) for linha in A) + "]"


def mostra(titulo, b, ce, cf, cv):
    """Duas políticas simples, as tabelas da recursão, o plano ótimo e a força bruta."""
    print("\n%s: b = %s, ce = %d, cf = %d, cv = %d" % (titulo, mat2str(b), ce, cf, cv))
    n, M = len(b), max(b)
    for xs, rot in [(b, "seguir as necessidades"), ([M] * n, "%d desde o início" % M)]:
        tot, cs = custo_plano(b, ce, cf, cv, xs)
        print("  %s %s: custos %s = %d" % (rot, mat2str(xs), mat2str(cs), tot))
    nplanos = int(np.prod([M - bt + 1 for bt in b]))
    print("  planos possíveis: %d" % nplanos)
    valor, plano, f, dec = mao_obra(b, ce, cf, cv, tabelas=True)
    print("  recursão para trás (s = operários na semana anterior):")
    for t in range(n - 1, -1, -1):
        for s in f[t]:
            cand = []
            for x in range(b[t], M + 1):
                c = ce * (x - b[t]) + ((cf + cv * (x - s)) if x > s else 0)
                cand.append("%d: %d+%d=%d" % (x, c, f[t + 1][x], c + f[t + 1][x]))
            print("    semana %d, s = %d: f = %-5d x* = %-6s [%s]"
                  % (t + 1, s, f[t][s], " ou ".join(map(str, dec[t][s])), "; ".join(cand)))
    tot, cs = custo_plano(b, ce, cf, cv, plano)
    print("  plano ótimo %s, custo %d (custos por semana %s)" % (mat2str(plano), valor, mat2str(cs)))
    cb, pb = mao_obra_bruta(b, ce, cf, cv)
    print("  força bruta (%d planos): %d com %s" % (nplanos, cb, mat2str(pb)))
    return valor, plano, f, cs, cb, pb, nplanos


ok = True
print("Complementos de IO — deck 5.3: gestão da mão-de-obra")

# ------------------------------------------------------------ montagem de eventos
b = [3, 8, 9, 4, 6]
valor, plano, f, cs, cb, pb, nplanos = mostra("Montagem de feiras e congressos", b, 300, 600, 150)
ok &= (valor == 3450 and plano == [3, 9, 9, 6, 6] and cs == [1050, 1800, 0, 600, 0]
       and cb == 3450 and pb == [(3, 9, 9, 6, 6)] and nplanos == 336
       and custo_plano(b, 300, 600, 150, b)[0] == 4050 and custo_plano(b, 300, 600, 150, [9] * 5)[0] == 6450
       and f[4] == {4: 900, 5: 750, 6: 0, 7: 0, 8: 0, 9: 0} and f[3] == {9: 600}
       and f[2] == {8: 1350, 9: 600} and f[1][3] == 2400 and f[0] == {0: 3450})

print("  e se o custo fixo de contratar mudar?")
sens = {}
for cf in (0, 300, 600, 1200, 2000):
    v = mao_obra(b, 300, cf, 150)[0]
    cbr, pbr = mao_obra_bruta(b, 300, cf, 150)
    sens[cf] = (v, pbr)
    print("    cf = %4d: custo %d, planos ótimos %s" % (cf, v, mat2str(pbr)))
ok &= (sens[0] == (1650, [(3, 8, 9, 4, 6)]) and sens[300][0] == 2850 and len(sens[300][1]) == 4
       and sens[600] == (3450, [(3, 9, 9, 6, 6)]) and sens[1200] == (4650, [(3, 9, 9, 6, 6)])
       and sens[2000] == (6050, [(9, 9, 9, 6, 6)]))

# ------------------------------------------------------------ Para resolver na aula
r7 = mostra("Exemplo 7", [5, 7, 8, 4, 6], 300, 400, 200)
r8 = mostra("Exemplo 8", [5, 7, 4, 6], 200, 300, 200)
ok &= r7[0] == 3300 and r7[1] == [5, 8, 8, 6, 6] and r7[5] == [(5, 8, 8, 6, 6)]
ok &= r8[0] == 2400 and r8[1] == [5, 7, 6, 6] and r8[5] == [(5, 7, 6, 6)]

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 5.3: gestão da mão-de-obra

Montagem de feiras e congressos: b = [3 8 9 4 6], ce = 300, cf = 600, cv = 150
  seguir as necessidades [3 8 9 4 6]: custos [1050 1350 750 0 900] = 4050
  9 desde o início [9 9 9 9 9]: custos [3750 300 0 1500 900] = 6450
  planos possíveis: 336
  recursão para trás (s = operários na semana anterior):
    semana 5, s = 4: f = 900   x* = 6      [6: 900+0=900; 7: 1350+0=1350; 8: 1800+0=1800; 9: 2250+0=2250]
    semana 5, s = 5: f = 750   x* = 6      [6: 750+0=750; 7: 1200+0=1200; 8: 1650+0=1650; 9: 2100+0=2100]
    semana 5, s = 6: f = 0     x* = 6      [6: 0+0=0; 7: 1050+0=1050; 8: 1500+0=1500; 9: 1950+0=1950]
    semana 5, s = 7: f = 0     x* = 6      [6: 0+0=0; 7: 300+0=300; 8: 1350+0=1350; 9: 1800+0=1800]
    semana 5, s = 8: f = 0     x* = 6      [6: 0+0=0; 7: 300+0=300; 8: 600+0=600; 9: 1650+0=1650]
    semana 5, s = 9: f = 0     x* = 6      [6: 0+0=0; 7: 300+0=300; 8: 600+0=600; 9: 900+0=900]
    semana 4, s = 9: f = 600   x* = 